# 01_pretrain

『밑바닥부터 시작하는 딥러닝 ❻』 실습 코드 — 원본: `ch03/01_pretrain.py`

셀을 위에서부터 차례대로 실행하세요.

In [ ]:
import os, sys

# 노트북에는 __file__이 없으므로 pyproject.toml이 있는 폴더(저장소 루트)를 찾아 이동한다
_dir = os.path.abspath('.')
while not os.path.exists(os.path.join(_dir, 'pyproject.toml')) and _dir != os.path.dirname(_dir):
    _dir = os.path.dirname(_dir)
os.chdir(_dir)
if '.' not in sys.path:
    sys.path.append('.')
print('작업 폴더:', os.getcwd())

In [ ]:
from itertools import cycle
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from tqdm import tqdm
from codebot.model import GPT
from codebot.utils import get_device

In [ ]:
# 설정
device = get_device()
data_path = 'codebot/tiny_codes.bin'
tokenizer_path = 'codebot/merge_rules.pkl'
model_save_path = 'codebot/model_pretrain.pt'

In [ ]:
# 하이퍼파라미터
context_len = 256
vocab_size = 1000
batch_size = 32
learning_rate = 3e-4
max_iters = 20000
embed_dim = 384
n_head = 6
n_layer = 6
ff_dim = 4 * embed_dim
dropout_rate = 0.1

In [ ]:
# 데이터셋 클래스
class TokenDataset(Dataset):
    def __init__(self, tokens, context_len):
        # 토큰열을 텐서로 변환하여 저장
        self.tokens = torch.tensor(tokens, dtype=torch.long)
        self.context_len = context_len

    def __len__(self):
        # 꺼낼 수 있는 샘플 수 반환
        return len(self.tokens) - self.context_len

    def __getitem__(self, idx):
        # 입력: idx번째부터 context_len개의 토큰
        x = self.tokens[idx:idx+self.context_len]
        # 레이블: 한 칸 뒤에서 시작하는 같은 길이의 토큰열
        y = self.tokens[idx+1:idx+self.context_len+1]
        return x, y

In [ ]:
# 데이터 준비
ids = np.fromfile(data_path, dtype=np.uint16)
dataset = TokenDataset(ids, context_len)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# 모델과 옵티마이저
model = GPT(
    vocab_size=vocab_size,
    max_context_len=context_len,
    embed_dim=embed_dim,
    n_head=n_head,
    n_layer=n_layer,
    ff_dim=ff_dim,
    dropout_rate=dropout_rate
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"파라미터 수: {total_params:,} ({total_params/1e6:.1f}M)")

In [ ]:
losses = []
data_iter = cycle(dataloader)  # 무한 반복
pbar = tqdm(range(max_iters))

In [ ]:
for i in pbar:
    batch_x, batch_y = next(data_iter)
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    logits = model(batch_x)
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), batch_y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    pbar.set_postfix({'loss': f'{loss.item():.4f}'})

In [ ]:
# 결과 저장
plt.figure(figsize=(10, 6))
plt.plot(losses)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.grid(True)
plt.savefig('loss_pretrain.png')

In [ ]:
# 모델 저장
model.save(model_save_path)